# Custom Containers

Package a job that needs more than the base image ships: build a custom container for the chemistry stack (OpenFermion, PySCF) and run a VQE inside it.

**Objectives:**
- Recognize when a job needs a custom container instead of the default Braket image
- Author a `Dockerfile` that extends the Braket base image with pinned dependencies
- Understand the `build_and_push.sh` flow: ECR login, `docker build`, `docker push`
- Pass `image_uri=...` to `AwsQuantumJob.create(...)` and know that in-container tasks still carry the job token (priority access, job-rate billing)
- Validate an ECR image URI before you ever spend a build

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first.
# Requires: pip install -e '.[dev]' from the project root (see `make setup`).
#
# Everything executed in this notebook runs locally with NO AWS credentials:
# we run a VQE on the LocalSimulator, inspect the Dockerfile/build script as
# strings, and validate an ECR URI with a regex. The docker build/push and the
# AwsQuantumJob.create call are shown as code and gated behind RUN_ON_AWS.

import re
import textwrap

import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator

# Importing the jobs/aws pieces is fine (no network). We only CALL them under RUN_ON_AWS.
from braket.aws import AwsQuantumJob  # noqa: F401  (used only inside if RUN_ON_AWS)

np.random.seed(7)

# Master switch. Left False so the notebook is safe to run end-to-end with no
# credentials, no Docker, no ECR, no cost. Flip to True only on a configured box.
RUN_ON_AWS = False

# Free, instant. The job we package would run this same loop on a managed instance.
device = LocalSimulator()
print("LocalSimulator ready. RUN_ON_AWS =", RUN_ON_AWS)

## 1. Why this job needs a custom container

The default Braket job image ships the SDK and common scientific packages, but it does **not** ship the quantum-chemistry stack from module 05 — OpenFermion (to build molecular Hamiltonians) and PySCF (the classical electronic-structure backend). A VQE-on-a-molecule job imports those at startup. If they are missing, the container dies before your algorithm runs.

The quickest way to know whether you need a custom container is to ask whether your entry-point script's imports resolve in the base image. We can sanity-check that idea right here: this notebook kernel is a stand-in for a bare environment, and the chemistry stack is not installed in it either.

In [ ]:
# The same import your VQE-on-a-molecule entry point would do at startup.
# In a bare environment (this kernel, or the default base image) it fails ->
# that failure is exactly the signal that you need a custom container.
try:
    import openfermion  # noqa: F401
    import openfermionpyscf  # noqa: F401
    HAVE_CHEM = True
except ImportError as exc:
    HAVE_CHEM = False
    missing = exc.name

if HAVE_CHEM:
    print("Chemistry stack present -- this environment could run the VQE as-is.")
else:
    print(f"Chemistry stack missing (no '{missing}').")
    print("A job importing it in the DEFAULT image would crash at startup.")
    print("=> Build a custom container that pip-installs the extra deps.")

## 2. The algorithm we are packaging (run it locally first)

Before worrying about containers, prove the algorithm works. The container is just a delivery vehicle for code you have already validated on the LocalSimulator. We use a small two-qubit, H2-like Hamiltonian in the computational basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$; its ground state lives in the single-excitation $\{|01\rangle, |10\rangle\}$ subspace, the classic VQE toy.

The ansatz is one fixed structure with a single free angle $\theta$ — the parametric-compilation pattern from notebook 02, and the literal inner loop a hybrid job would iterate:

$$|\psi(\theta)\rangle = \cos(\theta/2)\,|01\rangle + \sin(\theta/2)\,|10\rangle$$

In [ ]:
# A compact H2-like Hamiltonian (Hartree units, illustrative values).
H = np.array([
    [ 0.20,  0.00,  0.00,  0.00],
    [ 0.00, -0.30,  0.18,  0.00],
    [ 0.00,  0.18, -0.30,  0.00],
    [ 0.00,  0.00,  0.00,  0.50],
])
exact_ground = float(np.linalg.eigvalsh(H)[0])
print(f"Exact ground-state energy (classical diagonalization): {exact_ground:.4f} Ha")

# One fixed circuit structure, one free parameter -> compile once, reuse with new theta.
theta = FreeParameter("theta")

def ansatz():
    # X on q1 prepares |01>; the CNOT-RY-CNOT sandwich is a Givens rotation
    # that mixes |01> and |10> by angle theta. state_vector() lets us read the
    # exact expectation with shots=0 (deterministic, fast, no sampling noise).
    c = Circuit().x(1).cnot(1, 0).ry(1, theta).cnot(1, 0)
    c.state_vector()
    return c

def energy(theta_val):
    """<psi(theta)| H |psi(theta)> via the simulator's exact state vector."""
    psi = np.array(
        device.run(ansatz(), shots=0, inputs={"theta": float(theta_val)}).result().values[0]
    )
    return float(np.real(np.conjugate(psi) @ H @ psi))

print("energy(0.0) =", round(energy(0.0), 4), " energy(pi/2) =", round(energy(np.pi / 2), 4))

## 3. Run the variational loop and watch it converge

This is the loop a hybrid job would run on a managed instance, logging each step with `log_metric`. Here it runs locally so you see a concrete convergence curve with no AWS. A finite-difference gradient drives a small descent toward the ground state.

In [ ]:
theta_val = 0.2     # starting angle
lr = 0.8            # step size
eps = 0.05          # finite-difference spacing
n_iter = 20

history = []
for step in range(n_iter):
    e = energy(theta_val)
    history.append(e)
    grad = (energy(theta_val + eps) - energy(theta_val - eps)) / (2 * eps)
    theta_val -= lr * grad  # this is where log_metric("energy", e, step) would go in a job

print(f"start energy: {history[0]:.4f} Ha")
print(f"final energy: {history[-1]:.4f} Ha   (exact: {exact_ground:.4f} Ha)")
print(f"converged to within {abs(history[-1] - exact_ground):.1e} Ha")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(history)), history, "o-", color="#2563eb", label="VQE energy")
ax.axhline(exact_ground, color="#dc2626", ls="--", label=f"exact ({exact_ground:.3f} Ha)")
ax.set_xlabel("iteration")
ax.set_ylabel("energy (Ha)")
ax.set_title("Local VQE convergence -- the loop a hybrid job would run")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. The Dockerfile

Now the packaging. A custom container starts `FROM` the Braket base jobs image — so the SDK, the metrics pipeline, and the job entry-point contract are all already in place — and adds only your extra dependencies. The repo's real Dockerfile lives at [`../containers/Dockerfile`](../containers/Dockerfile); we reproduce its shape here as a string so we can read it without Docker installed.

Key points, line by line:
- **`FROM 292282985366.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-base-jobs:latest`** — the AWS-published Braket base image (account `292282985366` is AWS's, region `us-east-1`). Everything the job runtime needs is already here.
- **`COPY requirements.lock`** then **`RUN pip install`** — add the chemistry stack from a *pinned lock file* so builds are reproducible (no surprise upgrades between today and next month).
- **`COPY lib/`** — bake in the workspace's shared library so the entry point can `import lib...`.

In [ ]:
DOCKERFILE = textwrap.dedent("""\
    # Extend the AWS-published Braket base jobs image (account 292282985366 is AWS's).
    FROM 292282985366.dkr.ecr.us-east-1.amazonaws.com/amazon-braket-base-jobs:latest

    # Install pinned dependencies for reproducible builds.
    # requirements.lock pins openfermion, openfermionpyscf, pyscf, pennylane, etc.
    COPY 06-hybrid-jobs/containers/requirements.lock /tmp/requirements.lock
    RUN pip install --no-cache-dir -r /tmp/requirements.lock && rm /tmp/requirements.lock

    # Bake in the workspace shared library so the entry point can `import lib...`.
    COPY lib/ /opt/ml/code/lib/
""")

print(DOCKERFILE)

# A couple of cheap structural assertions -- these RUN (they are just string checks).
assert DOCKERFILE.lstrip().startswith("#") or DOCKERFILE.lstrip().startswith("FROM")
assert "amazon-braket-base-jobs" in DOCKERFILE, "must extend the Braket base image"
assert "pip install" in DOCKERFILE, "must install the extra dependencies"
print("Dockerfile looks structurally sound (extends base image, installs deps).")

## 5. The build-and-push script

With a Dockerfile in hand, `build_and_push.sh` ([`../containers/build_and_push.sh`](../containers/build_and_push.sh)) does four things: derive the ECR image URI from your account, log Docker in to ECR, `docker build`, and `docker push`. We show it as a string and walk through it — **none of it executes here** (no Docker, no AWS). You would run it from a shell on a configured machine.

The four steps:
1. **Resolve the URI** — `aws sts get-caller-identity` gives your account ID; combined with region and repo name it forms `<acct>.dkr.ecr.<region>.amazonaws.com/<repo>:<tag>`.
2. **ECR login** — `aws ecr get-login-password | docker login ...` for *both* your repo and the Braket base-image repo (the base lives in a public AWS ECR).
3. **Build** — `docker build -f .../Dockerfile -t <uri> .` from the project root, so `lib/` is in the build context.
4. **Push** — `docker push <uri>` uploads the image; that URI is what you hand to `AwsQuantumJob.create`.

In [ ]:
BUILD_AND_PUSH = textwrap.dedent("""\
    #!/usr/bin/env bash
    set -euo pipefail

    REGION="${AWS_DEFAULT_REGION:-us-east-1}"
    ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
    REPO_NAME="braket-quantum-workspace"
    IMAGE_TAG="latest"
    FULL_NAME="${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/${REPO_NAME}:${IMAGE_TAG}"

    # 1. Create the ECR repo if it does not exist yet.
    aws ecr describe-repositories --repository-names "$REPO_NAME" 2>/dev/null || \\
        aws ecr create-repository --repository-name "$REPO_NAME"

    # 2. Log Docker in to YOUR ECR and to the public Braket base-image ECR.
    aws ecr get-login-password --region "$REGION" | \\
        docker login --username AWS --password-stdin "${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com"
    aws ecr get-login-password --region us-east-1 | \\
        docker login --username AWS --password-stdin 292282985366.dkr.ecr.us-east-1.amazonaws.com

    # 3. Build from project root so lib/ is in the build context.
    cd "$(dirname "$0")/../.."
    docker build -f 06-hybrid-jobs/containers/Dockerfile -t "$FULL_NAME" .

    # 4. Push -- this URI is what AwsQuantumJob.create(image_uri=...) consumes.
    docker push "$FULL_NAME"
    echo "Image URI: $FULL_NAME"
""")

print(BUILD_AND_PUSH)
print("Shown only. To actually build/push, run ../containers/build_and_push.sh in a shell")
print("on a machine with Docker + AWS credentials. It is never invoked from this notebook.")

## 6. Validate the image URI before you spend a build

A `docker build` + `docker push` of the chemistry stack is minutes of work and bandwidth. A typo in the image URI fails the job *after* all that. Cheap insurance: validate the URI shape with a regex first. The ECR form is:

```
<12-digit-account>.dkr.ecr.<region>.amazonaws.com/<repository>:<tag>
```

In [ ]:
ECR_URI = re.compile(
    r"^(?P<account>\d{12})"                 # 12-digit AWS account id
    r"\.dkr\.ecr\."
    r"(?P<region>[a-z]{2}-[a-z]+-\d)"        # e.g. us-east-1, eu-west-2
    r"\.amazonaws\.com/"
    r"(?P<repo>[a-z0-9][a-z0-9._/-]*)"        # repository name
    r":(?P<tag>[\w.-]+)$"                     # image tag
)

def validate_image_uri(uri):
    m = ECR_URI.match(uri)
    if not m:
        return False, None
    return True, m.groupdict()

candidates = [
    "123456789012.dkr.ecr.us-east-1.amazonaws.com/braket-quantum-workspace:latest",
    "999900001111.dkr.ecr.eu-west-2.amazonaws.com/chem-jobs:v2.1",
    "12345.dkr.ecr.us-east-1.amazonaws.com/repo:latest",   # bad: account too short
    "braket-quantum-workspace:latest",                       # bad: not an ECR URI
]

for uri in candidates:
    ok, parts = validate_image_uri(uri)
    if ok:
        print(f"VALID    {parts['account']} | {parts['region']} | {parts['repo']}:{parts['tag']}")
    else:
        print(f"INVALID  {uri}")

## 7. The entry point and the gated `create` call

The job's entry point is just the VQE loop from sections 2-3, wrapped so it logs metrics and saves a result. We define it as a string (it would be written to `vqe_chemistry_job.py` and copied into the container) and then show the `AwsQuantumJob.create(image_uri=...)` call **gated behind `RUN_ON_AWS`**.

Two things to internalize:
- The only thing the custom image changes about `create` is the `image_uri` argument. Everything else — device ARN, source module, hyperparameters — is identical to a default-image job.
- **Tasks submitted from inside your container still carry the job token.** They get priority device access and are billed at *job rates*, not as standalone tasks. The custom image changes your software environment, not the job's billing or priority semantics.

In [ ]:
ENTRY_POINT = textwrap.dedent('''\
    # vqe_chemistry_job.py -- runs INSIDE the custom container, where openfermion/pyscf exist.
    import os
    from braket.aws import AwsDevice
    from braket.jobs import save_job_result
    from braket.jobs.metrics import log_metric
    # These imports are exactly why we need the custom image:
    from openfermion import MolecularData          # noqa: F401
    from openfermionpyscf import run_pyscf          # noqa: F401

    def main():
        device = AwsDevice(os.environ["AMZN_BRAKET_DEVICE_ARN"])
        # ... build the molecular Hamiltonian with openfermion/pyscf ...
        # ... define the FreeParameter ansatz (one fixed structure) ...
        theta = 0.2
        for step in range(100):
            e = evaluate_energy(device, theta)        # submits a priority task (job token)
            log_metric(metric_name="energy", value=e, iteration_number=step)
            theta = update(theta, e)                  # classical optimizer step
        save_job_result({"final_energy": e, "theta": theta})

    if __name__ == "__main__":
        main()
''')
print(ENTRY_POINT)

In [ ]:
# The custom-image job. Shown as code; only created when RUN_ON_AWS is True.
image_uri = "123456789012.dkr.ecr.us-east-1.amazonaws.com/braket-quantum-workspace:latest"

# Validate before we would ever pay to submit (re-using the section 6 check).
ok, _ = validate_image_uri(image_uri)
assert ok, "refusing to submit: image_uri is malformed"

if RUN_ON_AWS:
    # COST NOTE: This launches a managed instance (~$0.10-$3.00/hour while it runs)
    # PLUS the usual per-task / per-shot quantum charges -- a custom image does NOT
    # change billing, only your software environment. Prototype on LocalSimulator first;
    # cap spend with a stopping_condition (maxRuntimeInSeconds). (Project rule: estimate before QPU.)
    job = AwsQuantumJob.create(
        device="arn:aws:braket:::device/quantum-simulator/amazon/sv1",
        source_module="vqe_chemistry_job.py",
        entry_point="vqe_chemistry_job:main",
        image_uri=image_uri,                 # <-- the ONLY custom-container-specific argument
        hyperparameters={"max_steps": "100"},
        stopping_condition={"maxRuntimeInSeconds": 3600},  # cap wall-clock -> cap cost
        wait_until_complete=False,
    )
    print("Submitted:", job.arn)
else:
    print("RUN_ON_AWS is False -- not creating a job (no Docker build, no ECR, no cost).")
    print(f"Would submit with image_uri = {image_uri}")

## Exercises

Four exercises build on the pieces above -- the Dockerfile string, the URI
validator, and the gated `create` call. Each exercise is a prompt with two hint
tiers (open **Hint 1** for the idea, **Hint 2** for the concrete approach), a
scaffold cell to fill in, and a check cell to run right after your attempt. Keep
`RUN_ON_AWS = False` unless you intend to spend.

### Exercise 1 — Extend the Dockerfile with a pinned dependency

Section 4 baked the chemistry stack into the image. Rebuild that idea from the
outside in: collect a list of pinned dependencies and assemble a Dockerfile that
installs all of them.

Define `deps`: a list of at least three pinned dependency strings (each of the
form `package==version`, including one beyond OpenFermion and PySCF). Define
`custom_dockerfile`: the Dockerfile text that starts from the Braket base jobs
image and installs every entry in `deps`.

<details><summary>Hint 1 — nudge</summary>

A reproducible image pins exact versions. The `DOCKERFILE` string in section 4
split into two ideas: a base image, then the extra dependencies. You are
rebuilding that second part from a list you control -- what has to end up in the
install line?

</details>
<details><summary>Hint 2 — approach</summary>

Join `deps` into a single `RUN pip install --no-cache-dir ...` command, then
wrap it between a `FROM ...amazon-braket-base-jobs...` line and a
`COPY lib/ ...` line. String concatenation or an f-string is enough -- no Docker
required.

</details>

In [ ]:
# Exercise 1: Assemble a custom Dockerfile from a list of pinned dependencies.
# Define: deps -- list of pinned "package==version" strings (>= 3, one beyond
#         OpenFermion/PySCF); custom_dockerfile -- Dockerfile text that extends
#         the Braket base jobs image and installs every entry in deps.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert isinstance(deps, list) and len(deps) >= 3, (
        "list at least three pinned dependencies, one beyond the chemistry stack"
    )
    assert all(isinstance(d, str) and "==" in d for d in deps), (
        "pin every dependency as package==version so builds are reproducible"
    )
    assert isinstance(custom_dockerfile, str), "custom_dockerfile should be the Dockerfile text"
    assert "amazon-braket-base-jobs" in custom_dockerfile, (
        "a custom image must still extend the Braket base jobs image"
    )
    assert "pip install" in custom_dockerfile, "the Dockerfile has to install your dependencies"
    for _d in deps:
        assert _d.split("==")[0] in custom_dockerfile, (
            "every package named in deps should appear in the install line"
        )

### Exercise 2 — Accept an image referenced by digest

Section 6 validated the `<account>...:<tag>` form. Production images are often
pinned by content digest instead -- `<account>...@sha256:<64-hex>` -- which is
immutable in a way a moving tag is not. Harden the validator to accept both.

Define `validate_image_ref(uri)`: a function returning `(ok, parts)`. For a tag
reference `parts` carries a `tag`; for a digest reference it carries a `digest`
(and no `tag`). A malformed URI -- for instance an account that is not 12 digits
-- returns `(False, None)`.

<details><summary>Hint 1 — nudge</summary>

A digest reference swaps the `:` + tag suffix for an `@sha256:` + hex suffix.
The account/region/repository prefix is identical either way -- only the tail
changes. Which suffix are you trying to match?

</details>
<details><summary>Hint 2 — approach</summary>

Write two patterns that share the account/region/repository prefix: one ending
in `:<tag>`, one ending in `@sha256:` followed by 64 hex characters. Try the
digest pattern first, fall back to the tag pattern, and return the matching
group dictionary (or `(False, None)` when neither matches).

</details>

In [ ]:
# Exercise 2: Validate ECR references by tag OR by @sha256: digest.
# Define: validate_image_ref(uri) -> (ok, parts). A tag ref yields parts with a
#         "tag"; a digest ref yields parts with a "digest" (and no "tag"); a
#         malformed URI yields (False, None).

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    _tagged = "123456789012.dkr.ecr.us-east-1.amazonaws.com/chem-jobs:v2.1"
    _digest = "123456789012.dkr.ecr.us-east-1.amazonaws.com/chem-jobs@sha256:" + "a" * 64
    _short = "12345.dkr.ecr.us-east-1.amazonaws.com/chem-jobs:latest"
    _plain = "chem-jobs:latest"

    _ok_t, _parts_t = validate_image_ref(_tagged)
    _ok_d, _parts_d = validate_image_ref(_digest)
    _ok_s, _ = validate_image_ref(_short)
    _ok_p, _ = validate_image_ref(_plain)

    assert _ok_t and _ok_d, "both a :tag and an @sha256: digest reference should validate"
    assert not _ok_s and not _ok_p, "a non-12-digit account or a bare name must be rejected"
    assert "tag" in _parts_t, "a tag reference should report its tag"
    assert "digest" in _parts_d, "a digest reference should report its digest"
    assert "tag" not in _parts_d, "a digest reference is not a tag reference"

### Exercise 3 — A cost guard for the job

A custom image does not change a job's economics: you still pay for the managed
instance by the hour and for the quantum tasks per-task and per-shot. Turn that
into a number you can check against a budget before you ever submit.

Define `estimate_job_cost(instance_per_hr, runtime_s, n_tasks, per_task, shots,
per_shot)` returning the total dollar cost. Define `budget`: a positive dollar
cap you choose. Define `estimated_cost`: the estimate for a configuration like
section 7's, and satisfy yourself that it fits under `budget`.

<details><summary>Hint 1 — nudge</summary>

Two independent contributions add up: instance time (a rate per hour times a
duration) and quantum work (a per-task charge plus a per-shot charge, once per
task). Watch the units -- the runtime arrives in seconds.

</details>
<details><summary>Hint 2 — approach</summary>

Instance cost is `instance_per_hr * runtime_s / 3600`. Quantum cost is
`n_tasks * per_task + n_tasks * shots * per_shot`. Return their sum, then call
the function with your section-7 numbers and compare the result to `budget`.

</details>

In [ ]:
# Exercise 3: A cost guard -- estimate a job's dollar cost, check it fits a budget.
# Define: estimate_job_cost(instance_per_hr, runtime_s, n_tasks, per_task, shots,
#         per_shot) -> total dollars; budget -- a positive dollar cap you pick;
#         estimated_cost -- the estimate for a section-7-like configuration.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert callable(estimate_job_cost), "estimate_job_cost should be a function"
    assert abs(estimate_job_cost(0, 0, 0, 0, 0, 0)) < 1e-9, "a job that does nothing costs nothing"
    _one_hr = estimate_job_cost(1.0, 3600, 0, 0.0, 0, 0.0)
    assert abs(_one_hr - 1.0) < 1e-9, "runtime is in seconds -- one hour on a $1/hr instance is $1.00"
    _ten = estimate_job_cost(1.0, 3600, 10, 0.30, 1000, 0.01)
    _twenty = estimate_job_cost(1.0, 3600, 20, 0.30, 1000, 0.01)
    assert _twenty > _ten, "more tasks should cost strictly more (per-task and per-shot both scale)"
    assert _ten > _one_hr, "task and shot charges add on top of the instance-hour cost"
    assert budget > 0, "choose a positive dollar budget before you would submit"
    assert estimated_cost > 0, "estimate the cost of your section-7 configuration"
    assert estimated_cost <= budget, "your configuration should fit under the budget you set"

### Exercise 4 — Assemble the gated submit

Everything comes together in the `create` call. On a configured machine you would
run `build_and_push.sh`, take the Image URI it prints, and submit the job. Here,
assemble that call as data and validate it -- without submitting -- so it is ready
the moment you flip `RUN_ON_AWS`.

Define `job_spec`: a dict of the keyword arguments you would pass to
`AwsQuantumJob.create(...)` for the custom-container VQE job. It must carry an
`image_uri` that passes the section-6 validator, the usual `device`,
`source_module`, and `entry_point`, and a `stopping_condition` that caps
`maxRuntimeInSeconds`. Do not call `create`.

<details><summary>Hint 1 — nudge</summary>

The custom image changes exactly one argument to `create` -- everything else is
identical to a default-image job. And since a runaway job bills by the hour, the
one guard you never omit is the wall-clock cap. Which keys encode those two
ideas?

</details>
<details><summary>Hint 2 — approach</summary>

Collect the section-7 arguments into a dict: `device` (the SV1 ARN),
`source_module`, `entry_point`, `image_uri` (a validated ECR URI), and
`stopping_condition={"maxRuntimeInSeconds": ...}`. Run the `image_uri` through
`validate_image_uri` yourself before trusting it. No AWS call -- the dict is the
deliverable.

</details>

In [ ]:
# Exercise 4: Assemble (do not submit) the gated custom-container job.
# Define: job_spec -- dict of AwsQuantumJob.create(...) keyword arguments with a
#         validated image_uri, a device/source_module/entry_point, and a
#         stopping_condition capping maxRuntimeInSeconds. Keep RUN_ON_AWS = False.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert isinstance(job_spec, dict), "job_spec collects the create() keyword arguments"
    assert "image_uri" in job_spec, "the custom container enters through image_uri"
    _ok, _ = validate_image_uri(job_spec["image_uri"])
    assert _ok, "the image_uri must pass the section-6 validation before you submit"
    for _key in ("device", "source_module", "entry_point"):
        assert _key in job_spec, "a job still needs a device ARN, a source module, and an entry point"
    _sc = job_spec.get("stopping_condition")
    assert isinstance(_sc, dict) and _sc.get("maxRuntimeInSeconds", 0) > 0, (
        "cap wall-clock cost with a stopping_condition on maxRuntimeInSeconds"
    )

## Summary

- The default Braket job image lacks specialized dependencies (here: OpenFermion + PySCF). A **custom container** is how you add them — and an import failure in a bare environment is the signal you need one.
- A custom container starts `FROM` the Braket **base jobs image** and installs only your extra deps from a **pinned lock file** for reproducible builds.
- `build_and_push.sh` resolves the ECR URI, logs Docker in to ECR, builds from the project root, and pushes. That image URI is the single new argument to `AwsQuantumJob.create(image_uri=...)`.
- **The custom image changes your software environment, not the job's economics:** tasks submitted from inside still carry the **job token**, so they keep priority device access and job-rate billing. The instance is still billed per hour, the quantum tasks at the usual per-task/per-shot rates.
- Validate the image URI (and estimate cost) *before* you build, push, or submit. The cheapest job is the one that converges fast and shuts down promptly.

**Next:** [`06-pennylane-jobs.ipynb`](06-pennylane-jobs.ipynb) — run a full PennyLane training loop as a hybrid job, logging the curve and retrieving optimal parameters.